In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
#clean dataset load
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telco_churn_clean.csv"
)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# target variable create
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
#check
y.value_counts

<bound method IndexOpsMixin.value_counts of 0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64>

In [6]:
# normalize
y.value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [7]:
#feature matrix x create
X = df.drop(
    columns=[
        "customerID",
        "Churn"
    ]
)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


In [8]:
# id put in variable  [if need]
customer_ids = df["customerID"].copy()

#### Train/Test Split

In [9]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    id_train,
    id_test
) = train_test_split(
    X,
    y,
    customer_ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### Split size verify

In [10]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


#### Training class distribution

In [11]:
train_distribution = (
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print(train_distribution)

Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64


In [12]:
y_train.value_counts().sort_index()

Churn
0    4139
1    1495
Name: count, dtype: int64

#### Test class distribution

In [13]:
test_distribution = (
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print(test_distribution)

Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [14]:
y_test.value_counts().sort_index()

Churn
0    1035
1     374
Name: count, dtype: int64

#### create Comparison table

In [15]:
class_distribution = pd.DataFrame({
    "Full Dataset (%)":
        y.value_counts(normalize=True).sort_index() * 100,

    "Training (%)":
        y_train.value_counts(normalize=True).sort_index() * 100,

    "Testing (%)":
        y_test.value_counts(normalize=True).sort_index() * 100
})

class_distribution.index = [
    "No Churn",
    "Churn"
]

class_distribution.round(2)

,Full Dataset (%),Training (%),Testing (%)
No Churn,73.46,73.46,73.46
Churn,26.54,26.54,26.54


#### Verify train/test overlap

In [16]:
overlap = set(id_train).intersection(
    set(id_test)
)

print("Customer overlap:", len(overlap))

Customer overlap: 0


#### add assertions

In [17]:
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert len(X_train) + len(X_test) == len(df)

assert set(id_train).isdisjoint(
    set(id_test)
)

assert abs(
    y_train.mean() - y_test.mean()
) < 0.01

print("Train/test split validation passed.")

Train/test split validation passed.


#### Split manifest save

In [18]:
train_manifest = pd.DataFrame({
    "customerID": id_train,
    "split": "train",
    "Churn": y_train
})

test_manifest = pd.DataFrame({
    "customerID": id_test,
    "split": "test",
    "Churn": y_test
})

split_manifest = pd.concat(
    [
        train_manifest,
        test_manifest
    ]
).sort_index()

split_manifest.head()

,customerID,split,Churn
0,7590-VHVEG,train,0
1,5575-GNVDE,train,0
2,3668-QPYBK,train,1
3,7795-CFOCW,train,0
4,9237-HQITU,train,1


In [19]:
MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_manifest.csv"
)

split_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("Saved:", MANIFEST_PATH)

Saved: c:\AI_Projects\customer-churn-ml-system\data\processed\split_manifest.csv
